In [2]:
!pip install pennylane


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.1 MB/s eta 0:00:00


In [53]:
import pennylane as qml
from pennylane import numpy as np

N = 12
J = 1.0
h = 0.5
layers = 4 #más N → más layers

dev = qml.device("default.qubit", wires=N)

Definimos el Hamiltoniano de Ising

In [54]:
def ising_hamiltonian(N, J, h):

    coeffs = []
    obs = []

    # interacción Z_i Z_{i+1}
    for i in range(N - 1):
        coeffs.append(-J)
        obs.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # campo externo Z_i
    for i in range(N):
        coeffs.append(-h)
        obs.append(qml.PauliZ(i))

    return qml.Hamiltonian(coeffs, obs)

H_C = ising_hamiltonian(N, J, h) #COSTE

Hamiltoniano mezclador

In [55]:
def mixer_hamiltonian(N):
    coeffs = [-1.0 for _ in range(N)]
    obs = [qml.PauliX(i) for i in range(N)]
    return qml.Hamiltonian(coeffs, obs)

H_M = mixer_hamiltonian(N)

Circuito QAOA

In [59]:
@qml.qnode(dev)
def qaoa_circuit(gamma, beta):

    # estado inicial uniforme
    for i in range(N):
        qml.Hadamard(wires=i)

    # capas QAOA
    for l in range(layers):

        # COST LAYER (Ising)
        qml.qaoa.cost_layer(gamma[l], H_C)

        # MIXER LAYER
        qml.qaoa.mixer_layer(beta[l], H_M)

    return qml.expval(H_C)

Función de coste (energía del sistema)

In [60]:
def cost(params):
    gamma = params[:layers]
    beta = params[layers:]
    return qaoa_circuit(gamma, beta)

Optimización (encontrar estado de mínima energía)

In [61]:
params = np.random.randn(2 * layers, requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.1)

for i in range(100):

    params = opt.step(cost, params)

    if i % 10 == 0:
        print(f"Iter {i} | Energía = {cost(params):.6f}")

Iter 0 | Energía = -4.642187
Iter 10 | Energía = -14.764869
Iter 20 | Energía = -16.830322
Iter 30 | Energía = -16.742261
Iter 40 | Energía = -16.913230
Iter 50 | Energía = -16.969877
Iter 60 | Energía = -16.990767
Iter 70 | Energía = -16.997912
Iter 80 | Energía = -16.999113
Iter 90 | Energía = -16.999637


SOLUCIÓN EXACTA


In [52]:
import numpy as np
import itertools
N = 12
J = 1.0
h = 0.5
def energy(spins, J, h):
    E = 0.0

    # interacción vecinos
    for i in range(N - 1):
        E += -J * spins[i] * spins[i + 1]

    # campo externo
    for i in range(N):
        E += -h * spins[i]

    return E
configs = list(itertools.product([-1, 1], repeat=N))
energies = []

for c in configs:
    e = energy(c, J, h)
    energies.append((e, c))
min_energy, best_config = min(energies, key=lambda x: x[0])

print("Energía mínima exacta:", min_energy)
print("Configuración óptima:", best_config)

Energía mínima exacta: -17.0
Configuración óptima: (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1)
